# 11 — Kaggle VSL: MediaPipe keypoints → graph cache

Notebook này **không chạy lại RTMPose**. Theo cách của nhánh `feat/vsl400-mediapipe-pose-transformer`, nó mở dataset như **một remote ZIP duy nhất**, đọc central directory và lấy byte-range của keypoint MediaPipe canonical. Không gọi Kaggle API một lần cho từng file. Mặc định notebook tải toàn bộ keypoint canonical để sao lưu, nhưng chỉ chọn top 70 lớp cho manifest và graph cache. Sau đó dữ liệu được chuyển thành tensor graph `[64, 75, 7]`.

Luồng giữ nguyên phần học: **Graph Encoder → Spatial Transformer → Temporal Transformer**. Chỉ extractor/layout đầu vào đổi từ COCO-WholeBody sang `33 pose + 21 tay trái + 21 tay phải`. Điểm thứ 76 chưa được data card định nghĩa nên không được dùng trong graph v1; file nguồn trên Drive vẫn giữ nguyên đầy đủ 76 điểm.

> Split công khai chỉ có train/test và manifest xử lý không công bố signer ID. Notebook giữ nguyên test, rồi tách validation có phân tầng từ train. Không trình bày validation này là signer-disjoint.

> Trước khi tải, tạo hai Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY` hoặc một secret `KAGGLE_API_TOKEN`, rồi bật **Notebook access**. Token chỉ được gửi trong request xin URL archive; nó không được gửi sang URL lưu trữ đã ký.

In [ ]:
#@title Cấu hình
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
KAGGLE_DATASET = 'nguyenanfms/vsl-vietnamese-sign-language-v2'  #@param {type:'string'}
KAGGLE_VERSION = 0  #@param {type:'integer'}
# KAGGLE_VERSION=0 dùng phiên bản public mới nhất, tránh lỗi 404 do version cũ bị gỡ.
DATASET_HANDLE = KAGGLE_DATASET if KAGGLE_VERSION == 0 else f'{KAGGLE_DATASET}/versions/{KAGGLE_VERSION}'
VERSION_TAG = 'latest' if KAGGLE_VERSION == 0 else f'v{KAGGLE_VERSION}'
KAGGLE_KEYPOINT_DIRECTORY = 'processed/processed/keypoints_splited'  #@param {type:'string'}
CLASS_COUNT = 70  #@param {type:'integer'}
MIN_OFFICIAL_TRAIN_SAMPLES = 40  #@param {type:'integer'}
VALIDATION_FRACTION = 0.20  #@param {type:'number'}
SEED = 42  #@param {type:'integer'}
DOWNLOAD_ALL_CANONICAL_KEYPOINTS = True  #@param {type:'boolean'}
SAVE_KEYPOINT_ARCHIVE_TO_DRIVE = True  #@param {type:'boolean'}
COPY_KEYPOINT_FOLDER_TO_DRIVE = False  #@param {type:'boolean'}
CONVERT_LIMIT = 0  #@param {type:'integer'}
CONTINUE_ON_ERROR = False  #@param {type:'boolean'}

assert CLASS_COUNT > 0
assert MIN_OFFICIAL_TRAIN_SAMPLES >= 2
assert 0 < VALIDATION_FRACTION < 1


In [ ]:
#@title Mount Google Drive và khai báo đường dẫn
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_mediapipe')
SOURCE_SCOPE = 'all_canonical' if DOWNLOAD_ALL_CANONICAL_KEYPOINTS else f'top{CLASS_COUNT}_min{MIN_OFFICIAL_TRAIN_SAMPLES}'
DRIVE_SOURCE_ROOT = DRIVE_ROOT / 'source'
DRIVE_KEYPOINT_ROOT = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}'
DRIVE_KEYPOINT_ARCHIVE = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}_{VERSION_TAG}.tar'
DRIVE_KEYPOINT_ARCHIVE_REPORT = DRIVE_SOURCE_ROOT / f'keypoints_splited_{SOURCE_SCOPE}_{VERSION_TAG}.json'
SUBSET_ROOT = DRIVE_ROOT / f'subsets/top{CLASS_COUNT}'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
BUILD_REPORT = SUBSET_ROOT / 'manifest_report.json'
PINNED_GRAPH_CONFIG = SUBSET_ROOT / 'mediapipe_graph_config.pinned.yaml'
GRAPH_ROOT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/cache'
GRAPH_REPORT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/report.json'
LOCAL_DOWNLOAD_ROOT = Path(f'/content/kaggle-vsl-keypoints-{SOURCE_SCOPE}')
LOCAL_REPO = Path('/content/silent-signal')

for path in (DRIVE_ROOT, DRIVE_SOURCE_ROOT, SUBSET_ROOT, GRAPH_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


In [ ]:
#@title Xác thực và đọc keypoint từ một remote ZIP
import os, subprocess, sys

# Kaggle yêu cầu xác thực để cấp signed URL cho archive.
from google.colab import userdata

def _read_colab_secret(name):
    try:
        return (userdata.get(name) or '').strip()
    except Exception:
        return ''

kaggle_username = _read_colab_secret('KAGGLE_USERNAME')
kaggle_key = _read_colab_secret('KAGGLE_KEY')
kaggle_api_token = _read_colab_secret('KAGGLE_API_TOKEN')
if kaggle_username and kaggle_key:
    kaggle_auth_mode = 'KAGGLE_USERNAME + KAGGLE_KEY'
elif kaggle_api_token:
    kaggle_auth_mode = 'KAGGLE_API_TOKEN'
else:
    raise RuntimeError(
        'Thiếu Kaggle Secrets. Hãy tạo KAGGLE_USERNAME + KAGGLE_KEY '
        'hoặc KAGGLE_API_TOKEN, bật Notebook access, rồi chạy lại cell.'
    )

if kaggle_auth_mode == 'KAGGLE_USERNAME + KAGGLE_KEY':
    os.environ.pop('KAGGLE_API_TOKEN', None)
    os.environ['KAGGLE_USERNAME'] = kaggle_username
    os.environ['KAGGLE_KEY'] = kaggle_key
else:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_api_token
print('Kaggle authentication: OK —', kaggle_auth_mode)

# Dùng cùng cơ chế remote ZIP/HTTP Range của nhánh
# feat/vsl400-mediapipe-pose-transformer. Kaggle API chỉ được gọi
# một lần để xin signed archive URL; không ListTreeDatasetFiles.
if not (LOCAL_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
        'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF, f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(LOCAL_REPO)], check=True)
# Editable .pth được tạo sau khi kernel khởi động; thêm src ngay để import được không cần restart.
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
print('Commit:', subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip())

LOCAL_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
fetch_classes = 0 if DOWNLOAD_ALL_CANONICAL_KEYPOINTS else CLASS_COUNT
fetch_command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.fetch_kaggle_vsl_mediapipe',
    '--output-root', str(LOCAL_DOWNLOAD_ROOT),
    '--classes', str(fetch_classes),
    '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
    '--workers', '6',
    '--dataset', KAGGLE_DATASET,
    '--version', str(KAGGLE_VERSION),
]
print('+', ' '.join(fetch_command), flush=True)
subprocess.run(fetch_command, check=True)
LOCAL_KEYPOINT_ROOT = LOCAL_DOWNLOAD_ROOT
npy_count = sum(1 for _ in LOCAL_KEYPOINT_ROOT.rglob('*.npy'))
if npy_count == 0:
    raise RuntimeError(f'Không tìm thấy .npy trong {LOCAL_KEYPOINT_ROOT}')
print('Downloaded directory:', LOCAL_KEYPOINT_ROOT)
print('Keypoint files:', f'{npy_count:,}')


## Lưu toàn bộ keypoint canonical sang Drive

Mặc định cell dưới đây đóng toàn bộ cây `train/` + `test/` canonical thành **một file `.tar` duy nhất** trên Drive. Cách này nhanh và ổn định hơn việc ghi hàng chục nghìn file nhỏ. Nó không sao chép 72 GB video thô và không lấy `processed_augmented`. Nếu cần xem từng file trực tiếp trên Drive, bật `COPY_KEYPOINT_FOLDER_TO_DRIVE`, nhưng thao tác đó sẽ chậm.

In [ ]:
#@title Đóng gói toàn bộ keypoint thành một file trên Drive
import json, shutil, tarfile, time

source_files = sorted(LOCAL_KEYPOINT_ROOT.rglob('*.npy'))
source_bytes = sum(path.stat().st_size for path in source_files)
archive_marker = {
    'dataset_handle': DATASET_HANDLE,
    'source_directory': KAGGLE_KEYPOINT_DIRECTORY,
    'scope': SOURCE_SCOPE,
    'source_files': len(source_files),
    'source_bytes': source_bytes,
    'complete': True,
}

if SAVE_KEYPOINT_ARCHIVE_TO_DRIVE:
    archive_ok = False
    if DRIVE_KEYPOINT_ARCHIVE.is_file() and DRIVE_KEYPOINT_ARCHIVE_REPORT.is_file():
        try:
            previous = json.loads(DRIVE_KEYPOINT_ARCHIVE_REPORT.read_text(encoding='utf-8'))
            archive_ok = (
                previous.get('complete') is True
                and previous.get('dataset_handle') == DATASET_HANDLE
                and previous.get('scope') == SOURCE_SCOPE
                and previous.get('source_files') == len(source_files)
                and previous.get('source_bytes') == source_bytes
                and DRIVE_KEYPOINT_ARCHIVE.stat().st_size > 0
            )
        except (OSError, ValueError):
            archive_ok = False
    if archive_ok:
        print('Archive đã hoàn chỉnh, bỏ qua:', DRIVE_KEYPOINT_ARCHIVE)
    else:
        partial = DRIVE_KEYPOINT_ARCHIVE.with_suffix(DRIVE_KEYPOINT_ARCHIVE.suffix + '.partial')
        if partial.exists():
            partial.unlink()
        started = time.perf_counter()
        with tarfile.open(partial, mode='w') as archive:
            for position, source in enumerate(source_files, start=1):
                archive.add(source, arcname=source.relative_to(LOCAL_KEYPOINT_ROOT), recursive=False)
                if position == 1 or position == len(source_files) or position % 1000 == 0:
                    elapsed = time.perf_counter() - started
                    rate = position / elapsed if elapsed else 0
                    eta = (len(source_files) - position) / rate / 60 if rate else 0
                    print(f'archive {position:,}/{len(source_files):,} | ETA={eta:.1f} min')
            fetch_report = LOCAL_KEYPOINT_ROOT / '_fetch_report.json'
            if fetch_report.is_file():
                archive.add(fetch_report, arcname='_fetch_report.json', recursive=False)
        partial.replace(DRIVE_KEYPOINT_ARCHIVE)
        archive_marker['archive_bytes'] = DRIVE_KEYPOINT_ARCHIVE.stat().st_size
        archive_marker['elapsed_seconds'] = round(time.perf_counter() - started, 1)
        DRIVE_KEYPOINT_ARCHIVE_REPORT.write_text(
            json.dumps(archive_marker, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print('Đã lưu một archive trên Drive:', DRIVE_KEYPOINT_ARCHIVE)
        print(json.dumps(archive_marker, ensure_ascii=False, indent=2))
else:
    print('Bỏ qua tạo archive theo cấu hình.')

if COPY_KEYPOINT_FOLDER_TO_DRIVE:
    copied = skipped = 0
    copied_bytes = 0
    started = time.perf_counter()
    for position, source in enumerate(source_files, start=1):
        relative = source.relative_to(LOCAL_KEYPOINT_ROOT)
        destination = DRIVE_KEYPOINT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size == source.stat().st_size:
            skipped += 1
        else:
            shutil.copy2(source, destination)
            copied += 1
            copied_bytes += source.stat().st_size
        if position == 1 or position == len(source_files) or position % 500 == 0:
            elapsed = time.perf_counter() - started
            rate = position / elapsed if elapsed else 0
            eta = (len(source_files) - position) / rate / 60 if rate else 0
            print(f'{position:,}/{len(source_files):,} | copied={copied:,} | skipped={skipped:,} | ETA={eta:.1f} min')
    persisted = sum(1 for _ in DRIVE_KEYPOINT_ROOT.rglob('*.npy'))
    marker = {
        'dataset_handle': DATASET_HANDLE,
        'source_directory': KAGGLE_KEYPOINT_DIRECTORY,
        'selection': SOURCE_SCOPE,
        'source_files': len(source_files),
        'persisted_files': persisted,
        'complete': persisted == len(source_files),
    }
    (DRIVE_KEYPOINT_ROOT / '_copy_report.json').write_text(
        json.dumps(marker, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    if not marker['complete']:
        raise RuntimeError(f'Copy chưa đủ: {marker}')
    print(json.dumps(marker, ensure_ascii=False, indent=2))
else:
    print('Không copy từng file nhỏ; dùng archive .tar ở trên.')


In [ ]:
#@title Xác nhận source code đã cài
assert (LOCAL_REPO / '.git').is_dir()
print('Commit:', subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
#@title Chọn lớp, giữ test và tạo validation
import importlib, sys
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import silent_signal.cli.prepare_kaggle_vsl_mediapipe as prepare_mediapipe_module
prepare_mediapipe_module = importlib.reload(prepare_mediapipe_module)
prepare_mediapipe_main = prepare_mediapipe_module.main
print('Preparation module:', prepare_mediapipe_module.__file__)

# Quét toàn bộ bản local; lệnh build bên dưới chỉ chọn top 70 lớp để train.
KEYPOINT_ROOT_FOR_CONVERSION = LOCAL_KEYPOINT_ROOT
arguments = [
    'build',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--labels', str(LABELS),
    '--selection', str(SELECTION),
    '--report', str(BUILD_REPORT),
    '--classes', str(CLASS_COUNT),
    '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
    '--validation-fraction', str(VALIDATION_FRACTION),
    '--seed', str(SEED),
    '--dataset-handle', DATASET_HANDLE,
]
print('+ ss-prepare-kaggle-vsl-mediapipe', ' '.join(arguments), flush=True)
return_code = prepare_mediapipe_main(arguments)
if return_code:
    raise RuntimeError(f'Manifest build thất bại với mã {return_code}.')
missing_outputs = [path for path in (MANIFEST, LABELS, SELECTION, BUILD_REPORT) if not path.is_file()]
if missing_outputs:
    raise RuntimeError(f'Lệnh build kết thúc nhưng thiếu output: {missing_outputs}')
print('Manifest build: OK —', MANIFEST)


In [ ]:
#@title Pin config bằng manifest vừa tạo
import hashlib, json, yaml
from collections import Counter
import csv

payload = yaml.safe_load((LOCAL_REPO / 'configs/preprocessing/mediapipe_holistic_75_t64.yaml').read_text())
manifest_sha = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
with MANIFEST.open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
payload['expected'] = {
    'manifest_sha256': manifest_sha,
    'clips': len(rows),
    'classes': len({int(row['class_index']) for row in rows}),
    'splits': dict(Counter(row['split'] for row in rows)),
}
PINNED_GRAPH_CONFIG.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
print(PINNED_GRAPH_CONFIG.read_text())


In [ ]:
#@title Chuyển MediaPipe .npy thành graph cache — có resume
import importlib, sys
repo_src = str(LOCAL_REPO / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import silent_signal.cli.prepare_kaggle_vsl_mediapipe as prepare_mediapipe_module
prepare_mediapipe_module = importlib.reload(prepare_mediapipe_module)
prepare_mediapipe_main = prepare_mediapipe_module.main

arguments = [
    'convert',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--config', str(PINNED_GRAPH_CONFIG),
    '--output-root', str(GRAPH_ROOT),
    '--report', str(GRAPH_REPORT),
    '--progress-every', '50',
]
if CONVERT_LIMIT > 0:
    arguments += ['--limit', str(CONVERT_LIMIT)]
if CONTINUE_ON_ERROR:
    arguments.append('--continue-on-error')
print('+ ss-prepare-kaggle-vsl-mediapipe', ' '.join(arguments), flush=True)
return_code = prepare_mediapipe_main(arguments)
if return_code:
    raise RuntimeError(f'Graph conversion thất bại với mã {return_code}.')
if not GRAPH_REPORT.is_file():
    raise RuntimeError(f'Lệnh convert kết thúc nhưng thiếu report: {GRAPH_REPORT}')
print('Graph conversion: OK —', GRAPH_REPORT)


In [ ]:
#@title Kiểm tra kết quả cuối
import json, numpy as np

report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
cache_files = sorted(GRAPH_ROOT.rglob('*.npz'))
if not cache_files:
    raise RuntimeError('Không có graph cache nào được tạo.')
with np.load(cache_files[0], allow_pickle=False) as sample:
    print('Sample cache:', cache_files[0])
    print('features:', sample['features'].shape)
    print('joint_mask:', sample['joint_mask'].shape)
    print('adjacency:', sample['adjacency'].shape)
print(json.dumps({key: value for key, value in report.items() if key != 'failures'}, indent=2))
print('Full MediaPipe archive on Drive:', DRIVE_KEYPOINT_ARCHIVE)
if COPY_KEYPOINT_FOLDER_TO_DRIVE:
    print('Browsable MediaPipe folder on Drive:', DRIVE_KEYPOINT_ROOT)
print('Graph cache on Drive:', GRAPH_ROOT)
print('Manifest:', MANIFEST)
print('Labels:', LABELS)


## Đầu ra

- `source/keypoints_splited_all_canonical_latest.tar`: toàn bộ `.npy [T,76,3]` canonical trong một file bền vững trên Drive.
- `subsets/top70/manifest.csv`: nhãn liên tục của 70 lớp, test công khai được giữ nguyên.
- `subsets/top70/selection.json`: lớp được chọn và số mẫu từng split.
- `graph/mediapipe_holistic_75_v1_t64/cache/`: tensor `[64,75,7]` cho Graph Encoder → Spatial Transformer → Temporal Transformer.

Notebook train chỉ cần đọc `manifest.csv`, `labels.json`, config đã pin và thư mục graph cache; không cần tải lại Kaggle hoặc chạy pose extractor.